In [2]:
import numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import pandas as pd
import seaborn as sns

In [3]:
data = pd.read_csv('individuals_dataset_cleaned.csv')
data.head()

,ID,CODGEO,SEX,AGE,DIPLOMA,PRO_CAT,NBPERS_HOUSE,NB_10,NB_11_17,NB_18_24,...,TWO_WHEELER,BIKE,ELECT_SCOOTER,NAVIGO_SUB,IMAGINER_SUB,OTHER_SUB_PT,BIKE_SUB,NSM_SUB,WEIGHT_INDIV,GPS_RECORD
0,10_2978,78092.0,0.0,41.0,0.0,4.0,2.0,1.0,0.0,0.0,...,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1856.206160,1.0
1,10_2980,75120.0,1.0,30.0,5.0,2.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1375.000372,1.0
2,10_2981,91326.0,1.0,38.0,5.0,2.0,2.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1231.812990,1.0
3,10_2982,91573.0,1.0,43.0,4.0,2.0,1.0,1.0,0.0,0.0,...,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,426.311616,1.0
4,10_2984,78073.0,0.0,39.0,5.0,2.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,843.194726,1.0


In [4]:
#listar colunas
data.columns

Index(['ID', 'CODGEO', 'SEX', 'AGE', 'DIPLOMA', 'PRO_CAT', 'NBPERS_HOUSE',
       'NB_10', 'NB_11_17', 'NB_18_24', 'NB_25_64', 'NB_65', 'PMR',
       'DRIVING_LICENCE', 'NB_CAR', 'TWO_WHEELER', 'BIKE', 'ELECT_SCOOTER',
       'NAVIGO_SUB', 'IMAGINER_SUB', 'OTHER_SUB_PT', 'BIKE_SUB', 'NSM_SUB',
       'WEIGHT_INDIV', 'GPS_RECORD'],
      dtype='object')

### Notas siguientes
- Ajusta K_FINAL tras revisar métricas + perfilado.
- Interpreta cada cluster: construir tabla cruzada de DIPLOMA y PRO_CAT por cluster.
- Guardar CSV con asignaciones para futuros modelos.
- Próximo paso: añadir variables de movilidad (cuando estén) y repetir.


In [ ]:
# Alternativa: GaussianMixture con selección por BIC
from sklearn.mixture import GaussianMixture

X_prepared = final_pipe.named_steps['prep'].transform(X)  # reutiliza transformador ajustado (evita data leakage)

bic_list = []
for k in range(2, 11):
    gmm = GaussianMixture(n_components=k, covariance_type='full', n_init=5, random_state=42)
    gmm.fit(X_prepared)
    bic_list.append({'k': k, 'bic': gmm.bic(X_prepared), 'aic': gmm.aic(X_prepared)})

bic_df = pd.DataFrame(bic_list)
bic_df


In [ ]:
# Elegir K (ajusta este valor tras ver la gráfica / tabla)
K_FINAL = 6  # <-- cambia manualmente si ves mejor opción
final_pipe = Pipeline([
    ('prep', preprocessor),
    ('clust', KMeans(n_clusters=K_FINAL, n_init=30, random_state=42))
])
final_labels = final_pipe.fit_predict(X)

data['CLUSTER_KMEANS'] = final_labels

# Guardar pipeline para reutilizar (opcional)
import joblib
joblib.dump(final_pipe, 'kmeans_cluster_pipeline.pkl')

# Perfilado ponderado
def weighted_profile(df, cluster_col, weight_col):
    w = df[weight_col]
    g = df.groupby(cluster_col)
    out = []
    for c, sub in g:
        w_sub = sub[weight_col]
        tot_w = w_sub.sum()
        row = {
            'cluster': c,
            'n_individuos': len(sub),
            'peso_total': float(tot_w),
            'pct_peso': float(tot_w / w.sum())
        }
        # medias numéricas
        for col in numeric_cols:
            row[f'{col}_mean'] = (sub[col] * w_sub).sum() / tot_w
        # proporciones binarias
        for col in binary_cols:
            row[f'{col}_pct1'] = (sub[col] * w_sub).sum() / tot_w
        out.append(row)
    return pd.DataFrame(out)

profile_df = weighted_profile(data, 'CLUSTER_KMEANS', weight_col)
profile_df.sort_values('pct_peso', ascending=False)


In [ ]:
# Visualizar métricas para elegir K
fig, ax1 = plt.subplots(figsize=(8,4))
color = 'tab:blue'
ax1.set_xlabel('k')
ax1.set_ylabel('Inertia', color=color)
ax1.plot(res_df['k'], res_df['inertia'], marker='o', color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Silhouette', color=color)
ax2.plot(res_df['k'], res_df['silhouette'], marker='s', color=color)
ax2.tick_params(axis='y', labelcolor=color)
plt.title('Inertia vs Silhouette')
plt.show()
res_df.sort_values('silhouette', ascending=False).head()


In [ ]:
# Preprocesamiento y prueba de distintos K
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

# Selección de features para clustering (excluye id, peso)
features_binary = binary_cols  # ya 0/1, igual podemos tratarlas como numéricas sin escalado extra
features_numeric = numeric_cols
features_categorical = cat_cols

# Pipelines de transformación
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
    # no escalamos binarios (0/1) para mantener interpretabilidad parcial
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, features_numeric),
    ('bin', binary_pipeline, features_binary),
    ('cat', categorical_pipeline, features_categorical)
], remainder='drop')

X = data[features_numeric + features_binary + features_categorical]

k_list = list(range(2, 13))
results = []
X_trans = None
for k in k_list:
    pipe = Pipeline([
        ('prep', preprocessor),
        ('clust', KMeans(n_clusters=k, n_init=20, random_state=42))
    ])
    labels = pipe.fit_predict(X)
    # Guardar primero para k elegido si queremos reproducir
    if k == 2:
        X_trans = pipe.named_steps['prep'].transform(X)  # guardar ejemplo
    sil = silhouette_score(pipe.named_steps['prep'].transform(X), labels)
    inertia = pipe.named_steps['clust'].inertia_
    results.append({'k': k, 'silhouette': sil, 'inertia': inertia})

res_df = pd.DataFrame(results)
res_df


In [ ]:
# Clasificación de columnas
id_col = 'ID'
weight_col = 'WEIGHT_INDIV'
# Columnas disponibles
cols = data.columns.tolist()

# Binarias (0/1) detectadas manualmente por semántica
binary_cols = ['SEX','PMR','DRIVING_LICENCE','TWO_WHEELER','BIKE','ELECT_SCOOTER',
               'NAVIGO_SUB','IMAGINER_SUB','OTHER_SUB_PT','BIKE_SUB','NSM_SUB','GPS_RECORD']

# Numéricas puras (contadores / cantidades / edad)
numeric_cols = ['AGE','NBPERS_HOUSE','NB_10','NB_11_17','NB_18_24','NB_25_64','NB_65','NB_CAR']

# Categórica ordinal / nivel educativo y categoría profesional (tratar como categórico primero)
cat_cols = ['DIPLOMA','PRO_CAT']

# Verificar tipos
print('Revisión rápida:')
for c in numeric_cols + binary_cols:
    print(c, data[c].dtype, 'n missing:', data[c].isna().sum())

print('\nCategorías ejemplo:')
for c in cat_cols:
    print(c, data[c].nunique(), 'categorías')
    print('Ejemplos:', data[c].dropna().unique()[:10])

# Comprobación de exclusiones
used = set([id_col, weight_col] + binary_cols + numeric_cols + cat_cols)
unused = [c for c in cols if c not in used]
print('\nColumnas no usadas por ahora:', unused)


# Objetivo: Clustering socio-demográfico (fase 1)
Agrupar 3337 individuos usando sólo columnas socio-demográficas. Más adelante añadiremos movilidad.

Pasos:
1. Clasificar columnas en: identificador, peso, numéricas, categóricas binarias/multiclase.
2. Preprocesar: imputación (mediana / modo), escalado numéricos, one-hot categóricas.
3. Probar KMeans para K=2..12 (inercia + silhouette) y elegir un K inicial (criterio combinación de codo + silhouette + interpretabilidad).
4. Ajustar modelo final, asignar cluster.
5. Perfilado ponderado (usar WEIGHT_INDIV) de cada cluster.
6. (Opcional) Comparar con GaussianMixture (BIC/AIC) para soft clusters.

Nota sobre WEIGHT_INDIV: es el factor de expansión (peso de encuesta). Usarlo para:
- Calcular proporciones representativas de la población.
- Calcular medias ponderadas. No usarlo dentro de KMeans (afecta escala); se aplica sólo al resumir.

Luego guardamos asignaciones para usar como feature en modelos posteriores.

Ejecuta las celdas en orden. Si falta alguna librería (scikit-learn), instalarla antes: pip install scikit-learn seaborn.
